# 01 CUST_IDのデータ揺らぎの解決
## 文字列の正規化（名寄せへの応用)
### データの揺らぎイメージ
    * C10001 と c10001 が混在
    * Ｃ１０００１（全角）と C10001（半角）
    * " C10001 "（前後にスペース）、"C10001\n"（改行コードの巻き込み）
    * 区切り文字の有無: C-10001、C_10001、C 10001
    * 本来 001001 なのに 1001 になる
    * 00000000、99999999、unknown、test_user
    * "NULL" や "NaN" という「文字列」
    * C10001_DELETED（退会済）、C10001_DUP（重複）、C10001_01（2回目の登録）

In [ ]:
# データの取得
from snowflake.snowpark.context import get_active_session
import polars as pl 

session = get_active_session()

df_pd = (
    session
        .sql("SELECT * FROM KAGGLE_CREDIT_CARD.PUBLIC.RAW")
        .to_pandas()
)
df_pl = pl.DataFrame(df_pd)
print(df_pl.shape)
display(df_pl.head())

In [ ]:
df_pl.filter(pl.col("CUST_ID").str.contains(r'^C[1-9][0-9]{4}$')).shape

## 汚いデータを作成する

In [ ]:
import random
import polars as pl

def add_random_noise_to_id(df: pl.DataFrame, target_col: str = "CUST_ID", new_col: str = "CUST_ID_DIRTY", noise_rate: float = 0.3) -> pl.DataFrame:
    """
    Polars DataFrameの特定のID列に対して、実務で発生しがちな「揺らぎ」をランダムに付与する関数
    
    :param df: 元のPolars DataFrame
    :param target_col: 揺らぎを与えたい綺麗なIDの列名
    :param new_col: 生成される汚いIDの新しい列名
    :param noise_rate: 各揺らぎパターンが発生する確率（0.0 〜 1.0）
    """
    
    # 半角英数字を全角に変換するヘルパー関数
    def _to_zenkaku(text: str) -> str:
        return "".join(chr(ord(c) + 0xFEE0) if c.isalnum() and ord(c) < 128 else c for c in text)

    # 1件の文字列に対してノイズを混ぜるコアロジック
    def _introduce_noise(cust_id: str) -> str:
        if not cust_id:
            return cust_id
        
        current_id = cust_id
        
        # 数字部分とアルファベット部分を抽出しておく
        prefix = "".join([c for c in cust_id if c.isalpha()])
        digits = "".join([c for c in cust_id if c.isdigit()])
        
        # --- パターンA: 構造そのものの変化（Cの消失、前ゼロ付与） ---
        if random.random() < noise_rate:
            if random.random() < 0.5:
                # 10001 のように数字部分だけのもの
                current_id = digits
            else:
                # 001001 のように頭に不要な0が入る（元の桁数より長くゼロ埋め）
                current_id = digits.zfill(len(digits) + 2)
        
        # --- パターンB: 区切り文字の追加（Cが残っている場合のみ） ---
        if current_id and current_id[0].isalpha() and random.random() < noise_rate:
            sep = random.choice(["-", "_", " "])
            current_id = f"{current_id[0]}{sep}{current_id[1:]}"
            
        # --- パターンC: 大文字・小文字の混在 ---
        if random.random() < noise_rate:
            # ランダムに文字単位で小文字化
            current_id = "".join([c.lower() if random.random() < 0.5 else c for c in current_id])
            
        # --- パターンD: サフィックス（履歴等）の付与 ---
        if random.random() < noise_rate:
            current_id = f"{current_id}_01"
            
        # --- パターンE: 全角化 ---
        if random.random() < noise_rate:
            current_id = _to_zenkaku(current_id)
            
        # --- パターンF: 空白や改行コードの巻き込み（最終処理） ---
        if random.random() < noise_rate:
            if random.random() < 0.5:
                current_id = f" {current_id} "  # 前後にスペース
            else:
                current_id = f"{current_id}\n"  # 改行の巻き込み
                
        return current_id

    # Polarsの map_elements を使用してPython関数を適用
    # (※古いPolarsバージョンでは apply メソッドになります)
    return df.with_columns(
        pl.col(target_col).map_elements(_introduce_noise, return_dtype=pl.String).alias(new_col)
    )

In [ ]:
df_pl = add_random_noise_to_id(df_pl, target_col="CUST_ID", new_col="CUST_ID_DIRTY", noise_rate=0.6)
df_pl = (
    df_pl
        .drop("CUST_ID")
        .with_columns(pl.col("CUST_ID_DIRTY").alias("CUST_ID"))
        .drop("CUST_ID_DIRTY")
)

## 揺らぎの調整開始

In [ ]:
# 理想の形式に入力されてないデータを確認する
# 理想の形式 : "C10001", Cは大文字, 数字は5桁頭は0以外
df_pl.filter(~pl.col("CUST_ID").str.contains(r'^C[1-9][0-9]{4}$')).shape

In [ ]:
print("00_値のどこかに空白を持つデータ")
print(df_pl.filter(pl.col("CUST_ID").str.contains(r"\s")).shape)
df_tmp = (
    df_pl.with_columns(pl.col("CUST_ID").str.replace_all(r"\s", "").alias("CUST_ID"))
)
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"\s")).shape)

In [ ]:
print("01_全角英語を含むデータ")
print(df_tmp.filter(pl.col("CUST_ID").str.contains("[^ -~｡-ﾟ]+")).shape)
df_tmp = (
    df_tmp.with_columns(
        pl.col("CUST_ID").str.normalize("NFKC").alias("CUST_ID")
    )
)
print(df_tmp.filter(pl.col("CUST_ID").str.contains("[^ -~｡-ﾟ]+")).shape)

In [ ]:
print("02_全角数字を含むデータ")
upper_num_list = ["０","１","２","３","４","５","６","７","８","９", "C", "C", "c", "c"]
lower_num_list = ["0","1","2","3","4","5","6","7","8","9", "C", "C", "C", "C"]
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"[０-９]+")).shape)
df_tmp = df_tmp.with_columns(
    pl.col("CUST_ID").str.replace_many(upper_num_list, lower_num_list)
)
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"[０-９]+")).shape)

In [ ]:
print(df_tmp.filter(~pl.col("CUST_ID").str.contains(r'^C[1-9][0-9]{4}$')).shape)
print("03_Cのあとに数字以外が入っているデータ")
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"^[Cc].+[1-9][0-9]{4}.*$")).shape)

df_tmp = (
    df_tmp.with_columns(
        pl.col("CUST_ID").str.replace_all(r"^([Cc]).+([1-9][0-9]{4}).*$", r"$1$2")
    )
)
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"^[Cc].+[1-9][0-9]{4}.*$")))

In [ ]:
print("04_頭にCがないデータ")
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"^[1-9][0-9]{4}_.*$")).shape)
df_tmp = (
    df_tmp.with_columns(
        pl
        .when(pl.col("CUST_ID").str.contains(r"^[1-9][0-9]{4}_.*$"))
        .then("C" + pl.col("CUST_ID"))
        .otherwise(pl.col("CUST_ID"))
        .alias("CUST_ID")
    )
)
df_tmp.filter(pl.col("CUST_ID").str.contains(r"^[1-9][0-9]{4}_.*$"))

In [ ]:
print(df_tmp.filter(~pl.col("CUST_ID").str.contains(r'^C[1-9][0-9]{4}$')).get_column("CUST_ID").shape)
print("05_お尻に_で区切り文字があるデータ")
print(df_pl.filter(pl.col("CUST_ID").str.contains(r"C[1-9][0-9]{4}_.*$")).shape)
df_tmp = (
    df_tmp.with_columns(pl.col("CUST_ID").str.replace_all(r"(C[1-9][0-9]{4})_.*$", r"$1").alias("CUST_ID"))
)
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"C[1-9][0-9]{4}_.*$")).shape)

In [ ]:
df_tmp.filter(~pl.col("CUST_ID").str.contains(r'^C[1-9][0-9]{4}$')).get_column("CUST_ID")
print("06_頭が0で始まるデータ")
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"^0[1-9][0-9]{4}.*$")).shape)
print("06_頭が00で始まるデータ")
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"^00[1-9][0-9]{4}.*$")).shape)
df_tmp = (
    df_tmp.with_columns(pl.col("CUST_ID").str.replace_all(r"^(00)([1-9][0-9]{4}).*$", "C$2").alias("CUST_ID"))
)
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"^00[1-9][0-9]{4}.*$")).shape)


In [ ]:
df_tmp.filter(~pl.col("CUST_ID").str.contains(r'^C[1-9][0-9]{4}$')).get_column("CUST_ID")
print("07_頭が1で始まるデータ")

print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"^1.*$")).shape)
df_tmp = (
    df_tmp.with_columns(
        pl
        .when(pl.col("CUST_ID").str.contains(r"^1.*$"))
        .then("C" + pl.col("CUST_ID"))
        .otherwise(pl.col("CUST_ID"))
        .alias("CUST_ID")
    )
)
print(df_tmp.filter(pl.col("CUST_ID").str.contains(r"^1.*$")).shape)

In [ ]:
df_tmp.filter(pl.col("CUST_ID").str.contains(r'^C[1-9][0-9]{4}$')).get_column("CUST_ID")

In [ ]:
!pip install neologdn